# ParFlow-CLM vs SNOTEL Comparison

Load model output from Notebook 2, fetch SNOTEL observations, and evaluate:
- Daily SWE time series (obs vs. model)
- Scatter plot with 1:1 line
- Metrics: NSE, RMSE, |bias|%, peak SWE bias, melt-out day delta
- CW3E precipitation vs. SNOTEL precipitation (forcing quality check)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import hf_hydrodata as hf
from pathlib import Path
from datetime import datetime

sys.path.insert(0, os.path.abspath(".."))
from helpers import read_site_config

# HydroData PIN (persists from Notebook 1)
hf.register_api_pin("your_email@example.com", "your_pin")


In [ ]:
# ============================================================
# Load Site Configuration
# ============================================================
static_write_dir = "../runs/snow/Berthoud_Summit_WY2024"

run_dir = Path(static_write_dir)
cfg = read_site_config(run_dir / "site_config.txt")

site_name  = cfg["site_name"]
water_year = int(cfg["water_year"])
start      = cfg["start"]
end        = cfg["end"]
triplet    = cfg["triplet"]
region     = cfg["region"]

print(f"Site: {site_name} ({region}), WY{water_year}, triplet={triplet}")


In [ ]:
# ============================================================
# Load Model SWE from CLM NetCDF
# ============================================================
clm_files = sorted(run_dir.glob("*.out.CLM.*.nc"))
if not clm_files:
    raise FileNotFoundError(f"No CLM output in {run_dir}. Run Notebook 2 first.")

clm_ds = xr.open_dataset(clm_files[0])
if "swe_out" not in clm_ds:
    raise KeyError(f"swe_out not in CLM output. Variables: {list(clm_ds.data_vars)}")

swe_h = clm_ds["swe_out"].values.flatten()
dt_start = datetime.strptime(start, "%Y-%m-%d")
hourly_dates = pd.date_range(dt_start, periods=len(swe_h), freq="h")

model_swe_hourly = pd.Series(swe_h, index=hourly_dates, name="model_swe_mm")
model_swe_daily  = model_swe_hourly.resample("D").mean()

print(f"Loaded {len(swe_h)} hourly SWE values ({len(model_swe_daily)} days)")
print(f"  Peak model SWE: {model_swe_daily.max():.0f} mm")


In [ ]:
# ============================================================
# Fetch SNOTEL SWE observations
# ============================================================
df = hf.get_point_data(
    dataset="snotel", variable="swe",
    temporal_resolution="daily", aggregation="sod",
    site_ids=[triplet],
    date_start=start, date_end=end,
)

if df is None or df.empty or triplet not in df.columns:
    raise RuntimeError(f"No SNOTEL SWE data for triplet {triplet}")

obs_dates = pd.to_datetime(df["date"])
obs_swe_daily = pd.Series(df[triplet].values.astype(float),
                          index=pd.DatetimeIndex(obs_dates),
                          name="obs_swe_mm").dropna()
# SNOTEL SWE is already in mm — do NOT apply any unit conversion
print(f"SNOTEL SWE: {len(obs_swe_daily)} daily values, peak={obs_swe_daily.max():.0f} mm")


In [ ]:
# ============================================================
# Compute metrics on common dates
# ============================================================
common = obs_swe_daily.index.intersection(model_swe_daily.index)
obs_c = obs_swe_daily.loc[common]
mod_c = model_swe_daily.loc[common]
mask = (~obs_c.isna()) & (~mod_c.isna())
obs_c = obs_c[mask]
mod_c = mod_c[mask]

# Basic error stats
bias  = float((mod_c - obs_c).mean())
rmse  = float(np.sqrt(((mod_c - obs_c) ** 2).mean()))
peak_obs = float(obs_c.max())
peak_mod = float(mod_c.max())
bias_pct = (peak_mod - peak_obs) / peak_obs * 100.0 if peak_obs > 0 else np.nan

# Nash-Sutcliffe
mu_o    = obs_c.mean()
ss_res  = ((mod_c - obs_c) ** 2).sum()
ss_tot  = ((obs_c - mu_o) ** 2).sum()
nse     = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

# Peak SWE dates
peak_obs_date = obs_c.idxmax()
peak_mod_date = mod_c.idxmax()
peak_date_delta = (peak_mod_date - peak_obs_date).days

# Melt-out date (first day SWE < 10 mm after peak)
def melt_out(s, threshold=10.0):
    spring = s[s.index >= pd.Timestamp(f"{water_year}-02-01")]
    if spring.empty:
        return None
    peak_idx = spring.idxmax()
    post = spring[spring.index >= peak_idx]
    below = post[post < threshold]
    return below.index[0] if not below.empty else None

mo_obs = melt_out(obs_c)
mo_mod = melt_out(mod_c)
mo_delta = (mo_mod - mo_obs).days if (mo_obs and mo_mod) else None

print(f"\n{'='*55}")
print(f"  {site_name} WY{water_year} — SWE Comparison")
print(f"{'='*55}")
print(f"  N days:            {len(obs_c)}")
print(f"  Mean bias:         {bias:+.1f} mm")
print(f"  RMSE:              {rmse:.1f} mm")
print(f"  NSE:               {nse:.3f}")
print(f"  Peak SWE obs:      {peak_obs:.0f} mm on {peak_obs_date.date()}")
print(f"  Peak SWE model:    {peak_mod:.0f} mm on {peak_mod_date.date()}")
print(f"  Peak |bias|%:      {abs(bias_pct):.1f}%")
print(f"  Peak date delta:   {peak_date_delta:+d} days")
if mo_delta is not None:
    print(f"  Obs melt-out:      {mo_obs.date()}")
    print(f"  Model melt-out:    {mo_mod.date()}")
    print(f"  Melt-out delta:    {mo_delta:+d} days")
else:
    print(f"  Melt-out:          not reached in this period")
print(f"{'='*55}")


In [ ]:
# ============================================================
# Time Series Plot — Daily SWE
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(obs_c.index, obs_c.values, color="black",  lw=1.2, label="SNOTEL Obs")
ax.plot(mod_c.index, mod_c.values, color="steelblue", lw=1.2, label="PF-CLM Model")

if mo_obs is not None:
    ax.axvline(mo_obs, color="black",   ls="--", lw=0.8, alpha=0.5)
if mo_mod is not None:
    ax.axvline(mo_mod, color="steelblue", ls="--", lw=0.8, alpha=0.5)

ax.set_ylabel("SWE [mm]")
ax.set_title(f"{site_name} WY{water_year} — Daily SWE  |  "
             f"NSE={nse:.2f}, peak|bias|={abs(bias_pct):.0f}%, "
             f"melt-out Δ={mo_delta if mo_delta is not None else 'NA'}d")
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Scatter Plot — Observed vs Modeled
# ============================================================
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(obs_c.values, mod_c.values, s=12, alpha=0.5, color="steelblue", edgecolors="none")

lim_max = max(obs_c.max(), mod_c.max()) * 1.1
ax.plot([0, lim_max], [0, lim_max], "k--", lw=1, label="1:1")
ax.set_xlim(0, lim_max)
ax.set_ylim(0, lim_max)
ax.set_xlabel("SNOTEL SWE [mm]")
ax.set_ylabel("PF-CLM SWE [mm]")
ax.set_title(f"{site_name} WY{water_year}  |  NSE={nse:.2f}, RMSE={rmse:.0f} mm")
ax.set_aspect("equal")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Forcing Quality — CW3E vs SNOTEL Precipitation
# ============================================================
# SNOTEL precip (daily totals, already mm) vs CW3E precip (cumulative).
# CW3E is typically high-biased in the mountains — check the scale.

forcing_path = run_dir / cfg["forcing_file"]
forcing_raw  = np.loadtxt(forcing_path)
forc_dates   = pd.date_range(start, periods=len(forcing_raw), freq="h")

# CW3E precip is mm/s -> convert to mm/hr then daily sum
cw3e_precip_hourly = pd.Series(forcing_raw[:, 2] * 3600, index=forc_dates, name="cw3e_precip_mm")
cw3e_precip_daily  = cw3e_precip_hourly.resample("D").sum()
cw3e_precip_cum    = cw3e_precip_daily.cumsum()

# SNOTEL precip
try:
    df = hf.get_point_data(
        dataset="snotel", variable="precipitation",
        temporal_resolution="daily", aggregation="sum",
        site_ids=[triplet],
        date_start=start, date_end=end,
    )
    if df is not None and not df.empty and triplet in df.columns:
        snotel_precip_daily = pd.Series(df[triplet].values.astype(float),
                                        index=pd.to_datetime(df["date"]),
                                        name="snotel_precip_mm").dropna()
        # SNOTEL precip is already in mm
        snotel_precip_cum = snotel_precip_daily.cumsum()
    else:
        snotel_precip_cum = None
except Exception as e:
    print(f"SNOTEL precip fetch failed: {e}")
    snotel_precip_cum = None

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cw3e_precip_cum.index, cw3e_precip_cum.values, color="red", lw=1.5, label="CW3E")
if snotel_precip_cum is not None:
    ax.plot(snotel_precip_cum.index, snotel_precip_cum.values, color="black",
            lw=1.5, label="SNOTEL")
    ratio = cw3e_precip_cum.iloc[-1] / snotel_precip_cum.iloc[-1] if snotel_precip_cum.iloc[-1] > 0 else np.nan
    ax.set_title(f"{site_name} WY{water_year} — Cumulative Precip  |  "
                 f"CW3E/SNOTEL = {ratio:.2f}")
else:
    ax.set_title(f"{site_name} WY{water_year} — CW3E Cumulative Precip (no SNOTEL precip)")

ax.set_ylabel("Cumulative precip [mm]")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

clm_ds.close()
